In [1]:
import pduutil as pu
import pduhistoryserver as hs
import pandas as pd

datafiles_map = dict(pu.datafiles_iterator())

In [2]:
metric_names = hs.get_metric_names()
cpu_metrics = hs.get_cpu_metrics()
memory_metrics = hs.get_memory_metrics()
disk_read_metrics = hs.get_disk_read_metrics()
disk_write_metrics = hs.get_disk_write_metrics()
net_read_metrics = hs.get_net_read_metrics()

RAW_DF_FILE = "stages_with_metrics.csv.gz"
force_recalc = False
try:
    if force_recalc:
        print("Forcing recalculation")
        raise Exception
    print(f"Reading {RAW_DF_FILE}")
    raw_df = pd.read_csv(RAW_DF_FILE, compression="gzip")
except:
    print(f"Generating {RAW_DF_FILE}")
    from pdumetrics import PowerMetrics
    pm = PowerMetrics("all_power_processed.csv.gz")
    raw_df = hs.get_stages_dataframe(datafiles_map, metric_names, pm)
    raw_df.to_csv(RAW_DF_FILE, compression="gzip")
raw_df

Reading stages_with_metrics.csv.gz


,Unnamed: 0,app,app_id,app_start,app_end,num_machines,vm_config,app_runtime,app_energy,app_power_avg,...,energy,power_avg,internal.metrics.executorCpuTime,internal.metrics.executorDeserializeCpuTime,internal.metrics.peakExecutionMemory,internal.metrics.input.bytesRead,internal.metrics.shuffle.read.localBytesRead,internal.metrics.output.bytesWritten,internal.metrics.shuffle.write.bytesWritten,internal.metrics.shuffle.read.remoteBytesRead
0,0,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,1447.061534,290.000000,174351478,625224313,0,0,0,0,0,0
1,1,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,14678.124946,324.600000,236809871403,5145825322,0,17282373363,0,0,0,0
2,2,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,1572.779557,315.000000,12718394733,576376243,0,21240005760,0,0,0,0
3,3,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,3084.628677,299.333333,52941417073,488676466,0,42480011520,0,0,0,0
4,4,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,1475.676508,295.500000,28036068194,466096812,0,21960008640,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12689,12689,terasort,application_1716601033286_0018,2024-05-25 04:31:32.001000+00:00,2024-05-25 04:37:34.787000+00:00,6,vms6cores,362.786,213475.432823,576.693333,...,127667.842774,579.888889,1045897385790,2854568471,54873554160,0,2695778291,30000000000,0,13765807941
12690,12690,terasort,application_1716601033286_0019,2024-05-25 04:40:03.901000+00:00,2024-05-25 04:47:11.754000+00:00,6,vms6cores,427.853,247828.858083,569.272727,...,61007.661903,575.090909,612318579526,5009372317,52546240512,30000000000,0,0,16461586232,0
12691,12691,terasort,application_1716601033286_0019,2024-05-25 04:40:03.901000+00:00,2024-05-25 04:47:11.754000+00:00,6,vms6cores,427.853,247828.858083,569.272727,...,165596.854082,570.440678,945646839172,2646400810,54873554160,0,2981478610,30000000000,0,13480107622
12692,12692,terasort,application_1716601033286_0020,2024-05-25 04:49:34.196000+00:00,2024-05-25 04:55:07.362000+00:00,6,vms6cores,333.166,191203.750818,561.652174,...,63794.707467,554.000000,691836268073,5347912269,52546240512,30000000000,0,0,16461586232,0


In [3]:
df = raw_df

In [4]:
df.columns

Index(['Unnamed: 0', 'app', 'app_id', 'app_start', 'app_end', 'num_machines',
       'vm_config', 'app_runtime', 'app_energy', 'app_power_avg', 'stage_id',
       'stage_name', 'stage_submission_time', 'stage_completion_time',
       'stage_runtime', 'energy', 'power_avg',
       'internal.metrics.executorCpuTime',
       'internal.metrics.executorDeserializeCpuTime',
       'internal.metrics.peakExecutionMemory',
       'internal.metrics.input.bytesRead',
       'internal.metrics.shuffle.read.localBytesRead',
       'internal.metrics.output.bytesWritten',
       'internal.metrics.shuffle.write.bytesWritten',
       'internal.metrics.shuffle.read.remoteBytesRead'],
      dtype='object')

In [5]:
df[["app", "app_runtime"]]

,app,app_runtime
0,kmeans,299.146
1,kmeans,299.146
2,kmeans,299.146
3,kmeans,299.146
4,kmeans,299.146
...,...,...
12689,terasort,362.786
12690,terasort,427.853
12691,terasort,427.853
12692,terasort,333.166


In [7]:
df["cpu"] = sum(df[m] for m in cpu_metrics)
df["memory"] = sum(df[m] for m in memory_metrics)
df["disk_read"] = sum(df[m] for m in disk_read_metrics)
df["disk_write"] = sum(df[m] for m in disk_write_metrics)
df["net_read"] = sum(df[m] for m in net_read_metrics)
df

,Unnamed: 0,app,app_id,app_start,app_end,num_machines,vm_config,app_runtime,app_energy,app_power_avg,...,internal.metrics.input.bytesRead,internal.metrics.shuffle.read.localBytesRead,internal.metrics.output.bytesWritten,internal.metrics.shuffle.write.bytesWritten,internal.metrics.shuffle.read.remoteBytesRead,cpu,memory,disk_read,disk_write,net_read
0,0,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,0,0,0,0,0,799575791,0,0,0,0
1,1,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,17282373363,0,0,0,0,241955696725,0,17282373363,0,0
2,2,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,21240005760,0,0,0,0,13294770976,0,21240005760,0,0
3,3,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,42480011520,0,0,0,0,53430093539,0,42480011520,0,0
4,4,kmeans,application_1716774568868_0001,2024-05-27 02:09:15.106000+00:00,2024-05-27 02:14:14.252000+00:00,3,vms3cores,299.146,87660.042757,286.870968,...,21960008640,0,0,0,0,28502165006,0,21960008640,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12689,12689,terasort,application_1716601033286_0018,2024-05-25 04:31:32.001000+00:00,2024-05-25 04:37:34.787000+00:00,6,vms6cores,362.786,213475.432823,576.693333,...,0,2695778291,30000000000,0,13765807941,1048751954261,54873554160,2695778291,30000000000,13765807941
12690,12690,terasort,application_1716601033286_0019,2024-05-25 04:40:03.901000+00:00,2024-05-25 04:47:11.754000+00:00,6,vms6cores,427.853,247828.858083,569.272727,...,30000000000,0,0,16461586232,0,617327951843,52546240512,30000000000,16461586232,0
12691,12691,terasort,application_1716601033286_0019,2024-05-25 04:40:03.901000+00:00,2024-05-25 04:47:11.754000+00:00,6,vms6cores,427.853,247828.858083,569.272727,...,0,2981478610,30000000000,0,13480107622,948293239982,54873554160,2981478610,30000000000,13480107622
12692,12692,terasort,application_1716601033286_0020,2024-05-25 04:49:34.196000+00:00,2024-05-25 04:55:07.362000+00:00,6,vms6cores,333.166,191203.750818,561.652174,...,30000000000,0,0,16461586232,0,697184180342,52546240512,30000000000,16461586232,0


In [36]:
from sklearn.datasets import make_regression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Standardize features
    ('regressor', LinearRegression())  # Linear regression model
])

# Perform 5-fold cross-validation
cv = KFold(n_splits=6, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, df[metric_names], df["energy"], cv=cv, scoring='r2')

# Print cross-validation results
print("Cross-Validation R² Scores for Each Fold:")
for i, score in enumerate(scores, 1):
    print(f"  Fold {i}: R² = {score:.4f}")

print(f"\nMean R² Score: {np.mean(scores):.4f}")
print(f"Standard Deviation of R² Scores: {np.std(scores):.4f}")

['internal.metrics.executorCpuTime', 'internal.metrics.executorDeserializeCpuTime', 'internal.metrics.peakExecutionMemory', 'internal.metrics.input.bytesRead', 'internal.metrics.shuffle.read.localBytesRead', 'internal.metrics.output.bytesWritten', 'internal.metrics.shuffle.write.bytesWritten', 'internal.metrics.shuffle.read.remoteBytesRead']
Cross-Validation R² Scores for Each Fold:
  Fold 1: R² = 0.8153
  Fold 2: R² = 0.7943
  Fold 3: R² = 0.7906
  Fold 4: R² = 0.7919
  Fold 5: R² = 0.8115
  Fold 6: R² = 0.7914

Mean R² Score: 0.7992
Standard Deviation of R² Scores: 0.0102


In [41]:
from sklearn.datasets import make_regression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Standardize features
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))  # Random Forest model
])

# Perform 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, df[metric_names], df["energy"], cv=cv, scoring='r2')

# Print cross-validation results
print("Cross-Validation R² Scores for Each Fold:")
for i, score in enumerate(scores, 1):
    print(f"  Fold {i}: R² = {score:.4f}")

print(f"\nMean R² Score: {np.mean(scores):.4f}")
print(f"Standard Deviation of R² Scores: {np.std(scores):.4f}")

# Fit the pipeline to the entire dataset to get feature importances
pipeline.fit(df[metric_names], df["energy"])

# Get feature importances from the Random Forest
feature_importances = pipeline.named_steps['regressor'].feature_importances_

# Display feature importances
print("\nFeature Importances:")
for name, importance in zip(metric_names, feature_importances):
    print(f"  {name}: {importance:.4f}")

Cross-Validation R² Scores for Each Fold:
  Fold 1: R² = 0.9650
  Fold 2: R² = 0.9524
  Fold 3: R² = 0.9663
  Fold 4: R² = 0.9684
  Fold 5: R² = 0.9490

Mean R² Score: 0.9602
Standard Deviation of R² Scores: 0.0079

Feature Importances:
  internal.metrics.executorCpuTime: 0.4012
  internal.metrics.executorDeserializeCpuTime: 0.0231
  internal.metrics.peakExecutionMemory: 0.3929
  internal.metrics.input.bytesRead: 0.0270
  internal.metrics.shuffle.read.localBytesRead: 0.0242
  internal.metrics.output.bytesWritten: 0.0000
  internal.metrics.shuffle.write.bytesWritten: 0.0232
  internal.metrics.shuffle.read.remoteBytesRead: 0.1085
